In [21]:
%pip install -q transformers datasets accelerate torch librosa soundfile
%pip install -q transformers accelerate jiwer librosa soundfile

In [22]:
import os
import re
import torch
import librosa

from transformers import AutoModelForSpeechSeq2Seq
from transformers import AutoProcessor

from jiwer import wer, cer

In [23]:
AUDIO_FOLDER = "/content/audio"

In [24]:
ground_truths = {
    1: "حاجة مُثبتة علميًا.نعرف إن هي موجود في الـScience يعني.",
    2: "بس هو يعني النحل هو في تلت انواع نحل في الخلية",
    3: "صح ولا لأ؟",
    4: "و Pilgrim’s Pride",
    5: "فانا بحب ا avoid إن أنا أتعامل مع حد أعلى مني أو اقل مني",
    6: "بحس ان هم فى attitude كده انا مش عارفه ا fake ه لإن بيبقى في fakeness شوية كثير",
    7: "يعني بقيت more confident و أن هو مش لازم لما تحصلي حاجة في حياتي أقف فاهمة و ماكملش و اللي هو لأ خلاص مش هتعرف علي ناس وكده لأ يعني ..",
    8: "هأعمل كهربا WiFi، قبل ما يبقى فيه WiFi.",
    9: "الدول اللي كان بيحاربها وكذبته،",
    10: "punctuality في الشغل بالنسبالي دكتور slim role model بصراحة لأن هو عارف هو بيعمل أيه كويس أوي",
    11: "أو Katsura Chemical اليابانية",
    12: "على إنها شركة بتسرق خير بلده، يشوف إن بينهم علاقة متبادلة.",
    13: "يعنى حاجات بتاعة ال event of charity",
    14: "أنا ماليش favorite movie الصراحة.",
    15: "دخلت media engineering and technology",
    16: "دى كان بالنسبالى انا، دي كشخصيتى يعنى كشخصيتى انا",
    17: "بس افضل، يا عزيزي، اعمل Mix بقى، عيش في حالة توتر ورعب، انت كامل!",
    18: "المهم، يا عزيزي، الفكرة في الـ دايت دا إن انت بتحافظ على الـKetosis طول الوقت،",
    19: "يعني أي حاجة مابقدرش أعملها، ببقى هتجنن، أنا أزاي مش هوصلها، لأ هوصلك، كده يعنى",
    20: "مجرد بس تسجيل دخول، Check-in.",
    21: "لدرجة، يا عزيزي، إن آبل حاولت في IOS 10 تغيّر Design الخوخة!",
    22: "و اللي احنا عاوزين نوصله",
    23: "أنتوا معاكوا like fifty، sixty، seventy percent من ال content بتاع ال test، مافضلش غير حاجة بسيطة",
    24: "تواضع",
    25: "النص الحقيقي للعينة الخامسة والعشرين"
}

In [25]:
tuned_transcription = {
    1: " حاجة مثبتة علميًا، نعرف إنها موجودة في الـScience، يعني،",
    2: "بس هو يعني النحلة هو فيه 3 انواع نحلة في الخلية",
    3: "صحة الله",
    4: "وبالجرام سبرايت",
    5: "فأنا بحب avoid إن أنا أتعمل محد أعلى مني أو أقلي مني",
    6: "بحس ان هما فيه attitude كده انا مش عارفها أفاكل إن بيبقى فيه fakeness شوية كتير",
    7: "يعني بقيت more confident و ان هو مش لازم لما تحصل لي حاجة في حياتي اقف فهمها و ماكملش ولهوا لأ خلص مش هتعرف على الناس و كده لأ",
    8: "هعمل كهرباء Wi-Fi قبل ما يبقى فيه Wi-fi.",
    9: "الدول اللي كان بيحاربها وكذبته،",
    10: "Punctuality في الشغل بالنسبالي Dr. Slim Rule model بصراحة لأنه عارف هو بيعمل إيه كويسي قوي",
    11: "أو كانت صورة Chemical اليابانية",
    12: "على إنها شركة بتسرق خير بلده، يشوف إن بينهم علاقة متبادلة.",
    13: "يعني حاجات بتارة ال events بتاعة ال charity",
    14: "ماليش favorite movie صراحة",
    15: "دخلت Media Engineering and Technology",
    16: "دي كان بالنسبالي أن دي كشخصيتي يعني كش خصياتي أنا",
    17: "بس افضل، يا عزيزي، يعمل الـMix داعش في حال التوتر و رعب، انت كامل.",
    18: "أولا، يا عزيزي، الفكرة في الـDiet دا، أنّ انت بتحفظ على كيتوزيس طول الوقت.",
    19: "يعني أي حاجة ما بقدرش أعملها ببقى حجنن أنا ازاي مش هوصلها، لأ هوصلي كده يعني",
    20: "مجرّد، بس، تسجيل دخول، Check-In.",
    21: "من درجة، يا عزيزي، إن أبل حولت في iOS 10، تغير Design الخوخ.",
    22: "واللي احنا ايزين نوصلله",
    23: "انتو معاكو like 50, 60, 70% من ال content بتاع التست مبتلس شارك الحاجة بسيطة",
    24: "تواضع",
    25: "فحسس ان هو شخص اداونيه داية فعشان كده أنا عايز.. يعني عشانكده أعايز أقرأه، أنه هو قالي كتاب دا حلو، مش عارف ايه، وقراوس يعني ياتليه هيفيدك"
}

In [26]:
model_id1 = "openai/whisper-base"

processor1 = AutoProcessor.from_pretrained(model_id1)

model1 = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id1,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to("cuda" if torch.cuda.is_available() else "cpu")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

In [27]:
model_id2 = "openai/whisper-large-v3-turbo"

processor2 = AutoProcessor.from_pretrained(model_id2)

model2 = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id2,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to("cuda" if torch.cuda.is_available() else "cpu")

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

In [28]:
def normalize_arabic(text):
    text = text.lower()

    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)

    # Remove Arabic diacritics
    text = re.sub(r"[\u064B-\u065F]", "", text)

    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [30]:
# Whisper Base
references1 = []
predictions1 = []

forced_prompt_ids = processor1.get_decoder_prompt_ids(
    language="arabic",
    task="transcribe"
)
model1.config.use_cache = True
print("Running quantitative evaluation...\n")
for i in range(1, 26):
    audio_path = os.path.join(AUDIO_FOLDER, f"{i}.wav")
    audio_data, sampling_rate = librosa.load(
        audio_path,
        sr=16000
    )
    inputs = processor1(
        audio_data,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).to(
        "cuda" if torch.cuda.is_available() else "cpu",
        dtype=model1.dtype
    )
    with torch.no_grad():
        predicted_ids = model1.generate(
            inputs.input_features,
            num_beams=5,
            forced_decoder_ids=forced_prompt_ids,
            max_new_tokens=255,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    prediction = processor1.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    ground_truth = ground_truths[i]

    normalized_reference = normalize_arabic(ground_truth)
    normalized_prediction = normalize_arabic(prediction)
    references1.append(normalized_reference)
    predictions1.append(normalized_prediction)

    print(f"\nSample {i}")
    print("Ground Truth:")
    print(normalized_reference)
    print("\nPrediction:")
    print(normalized_prediction)
    print("\n" + "=" * 60)

Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running quantitative evaluation...



Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 1
Ground Truth:
حاجه مثبته علميانعرف ان هي موجود في الـscience يعني

Prediction:
حاجه مزبت علي المين نعرف انه يمود في السنس يعني



Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Sample 21
Ground Truth:
لدرجه يا عزيزي ان ابل حاولت في ios 10 تغير design الخوخه

Prediction:
ناجت عزيجي نابل حول الفايق ويستعشها تغير دزين الخوخه



Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 22
Ground Truth:
و اللي احنا عاوزين نوصله

Prediction:
واليحنا يزينوا صلوف



Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 23
Ground Truth:
انتوا معاكوا like fifty sixty seventy percent من ال content بتاع ال test مافضلش غير حاجه بسيطه

Prediction:
في المعكوه like 5670 من the content that test من the share of haga basito


Sample 24
Ground Truth:
تواضع

Prediction:
توضو


Sample 25
Ground Truth:
النص الحقيقي للعينه الخامسه والعشرين

Prediction:
فحسس نوع شخص دوني ديه فشنكدا عايز يعني عشقا عز قراء انها هو اليك كتاب ده حلوم شرفيه وقراوس يعني اتليها يفيدك



In [31]:
# Whisper Large V3 Turbo
references2 = []
predictions2 = []

forced_prompt_ids = processor2.get_decoder_prompt_ids(
    language="arabic",
    task="transcribe"
)
model2.config.use_cache = True
print("Running quantitative evaluation...\n")
for i in range(1, 26):
    audio_path = os.path.join(AUDIO_FOLDER, f"{i}.wav")
    audio_data, sampling_rate = librosa.load(
        audio_path,
        sr=16000
    )
    inputs = processor2(
        audio_data,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).to(
        "cuda" if torch.cuda.is_available() else "cpu",
        dtype=model2.dtype
    )
    with torch.no_grad():
        predicted_ids = model2.generate(
            inputs.input_features,
            num_beams=5,
            forced_decoder_ids=forced_prompt_ids,
            max_new_tokens=255,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    prediction = processor2.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    ground_truth = ground_truths[i]

    normalized_reference = normalize_arabic(ground_truth)
    normalized_prediction = normalize_arabic(prediction)
    references2.append(normalized_reference)
    predictions2.append(normalized_prediction)

    print(f"\nSample {i}")
    print("Ground Truth:")
    print(normalized_reference)
    print("\nPrediction:")
    print(normalized_prediction)
    print("\n" + "=" * 60)

Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running quantitative evaluation...



Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 1
Ground Truth:
حاجه مثبته علميانعرف ان هي موجود في الـscience يعني

Prediction:
حاجه مثبته علميا نعرف ان هي موجوده في الساينس يعني



Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Sample 22
Ground Truth:
و اللي احنا عاوزين نوصله

Prediction:
وليح نيزي نوصله



Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 23
Ground Truth:
انتوا معاكوا like fifty sixty seventy percent من ال content بتاع ال test مافضلش غير حاجه بسيطه

Prediction:
انتو معاكو like 506070 من الكلام بتاع التست تشارك بشيء بسيطه



Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 24
Ground Truth:
تواضع

Prediction:
دو دو


Sample 25
Ground Truth:
النص الحقيقي للعينه الخامسه والعشرين

Prediction:
فحسس ان هو شخص الدوني هدايف علشان كده عايز اقراء انه هو قالك تبدا حلو مش عارف ايه اقراوس يعني قلت ليه هيفيدك



In [32]:
# Our Model
references = []
predictions = []

for i in range(1, 26):
    references.append(normalize_arabic(ground_truths[i]))
    predictions.append(normalize_arabic(tuned_transcription[i]))

In [36]:
overall_wer = wer(references, predictions)
overall_cer = cer(references, predictions)
overall_wer1 = wer(references1, predictions1)
overall_cer1 = cer(references1, predictions1)
overall_wer2 = wer(references2, predictions2)
overall_cer2 = cer(references2, predictions2)

print("\n==================================")
print("Tuned Whisper Model RESULTS")
print("==================================")
print(f"WER: {overall_wer}")
print(f"CER: {overall_cer}")
print("==================================")

print("\n==================================")
print("Whisper Large V3 Turbo RESULTS")
print("==================================")
print(f"WER: {overall_wer2}")
print(f"CER: {overall_cer2}")
print("==================================")

print("\n==================================")
print("Whisper Base RESULTS")
print("==================================")
print(f"WER: {overall_wer1}")
print(f"CER: {overall_cer1}")
print("==================================")


Tuned Whisper Model RESULTS
WER: 0.4879032258064516
CER: 0.24575586095392077

Whisper Large V3 Turbo RESULTS
WER: 0.6895161290322581
CER: 0.4211802748585287

Whisper Base RESULTS
WER: 0.9112903225806451
CER: 0.5666936135812449
